In [ ]:
import os
import shutil
from pathlib import Path
import cv2
import pytesseract
from matplotlib import pyplot as plt

# ---------------- Configuration ----------------
# Match working folder structure
CURRENT_DIR = Path.cwd()
IMAGE_DIR = CURRENT_DIR / "images" if (CURRENT_DIR / "images").is_dir() else CURRENT_DIR

# Exact filenames present in your images directory
TARGET_IMAGES = [
    "diplomatic.jpg",
    "Evehicle.jpg",
    "government.jpg",
    "higherups.jpg",
    "private.jpg",
    "public.jpg"
]

# Crop region as fractions of the image: (top, bottom, left, right)
CROP_BOX = (0.12, 0.90, 0.025, 0.975)
PADDING = 25
TESSERACT_CONFIG = r"--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

TESSERACT_CANDIDATES = [
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
    os.path.expandvars(r"%LOCALAPPDATA%\Programs\Tesseract-OCR\tesseract.exe"),
    os.path.expandvars(r"%PROGRAMFILES%\Tesseract-OCR\tesseract.exe"),
]


# ---------------- Tesseract setup ----------------
def setup_tesseract():
    """Find tesseract.exe (PATH first, then common folders) and configure pytesseract."""
    candidates = [shutil.which("tesseract")] + TESSERACT_CANDIDATES
    for path in candidates:
        if path and os.path.exists(path):
            pytesseract.pytesseract.tesseract_cmd = path
            print(f"Using Tesseract {pytesseract.get_tesseract_version()} at: {path}")
            return path
    raise RuntimeError(
        "Tesseract executable not found. Please install Tesseract OCR or set its path properly."
    )


# ---------------- Processing Steps ----------------
def load_image(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not load '{path}'. Current dir: {os.getcwd()}")
    return img


def crop_plate_line(img, box=CROP_BOX):
    h, w = img.shape[:2]
    top, bottom, left, right = box
    return img[int(h * top):int(h * bottom), int(w * left):int(w * right)]


def preprocess(crop, padding=PADDING):
    """Grayscale -> Otsu threshold -> black text on white -> white padding."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Ensure black characters on white background
    if cv2.countNonZero(thresh) < thresh.size / 2:
        thresh = cv2.bitwise_not(thresh)

    return cv2.copyMakeBorder(
        thresh, padding, padding, padding, padding, cv2.BORDER_CONSTANT, value=255
    )


def read_text(processed):
    return pytesseract.image_to_string(processed, config=TESSERACT_CONFIG).strip()


def show_comparison(original, processed, title=""):
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    axs[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axs[0].set_title(f"Original {title}".strip())
    axs[1].imshow(processed, cmap="gray")
    axs[1].set_title("Processed for OCR")
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# ---------------- Pipeline Execution ----------------
def ocr_image(image_path, show=True):
    img = load_image(image_path)
    processed = preprocess(crop_plate_line(img))
    text = read_text(processed)
    if show:
        show_comparison(img, processed, title=f"({os.path.basename(image_path)})")
    return text


def run_all(image_files=TARGET_IMAGES, image_dir=IMAGE_DIR):
    results = {}
    for filename in image_files:
        path = image_dir / filename
        try:
            results[filename] = ocr_image(path)
            print(f"[{filename}] Extracted Text: {results[filename]}\n")
        except FileNotFoundError as e:
            results[filename] = None
            print(f"[{filename}] SKIPPED: {e}\n")
    return results


# ---------------- Run ----------------
setup_tesseract()
results = run_all()